# 11 · Extensions: activation functions, and the paper's open questions

**Paper:** Comments & Replications (Redwood Research), p. 51; Open Questions, pp. 45–46. Also
pp. 13–16 (the phase change this notebook varies).
**Experiment:** E12 `activation_functions` (an extension: no pass/fail).

**Learning goals**

1. See why the output nonlinearity sets the price of interference: the ReLU makes negative
   interference free (p. 13); other activations do not.
2. Compute a weight family's exact expected loss for any activation, and compare activations on
   the same task.
3. Read E12's diagrams, and tell what the loss landscape prefers (the theory) from what training
   reaches (the vote of trained models).
4. Pick a first extension project from the paper's open questions.

The E12 quick profile took 548 s on an Apple M2 CPU (six activations × 640 models).

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from superposition import theory
from superposition.experiments.activation_functions import (
    DEFAULT_ACTIVATIONS,
    ActivationFunctionsConfig,
)
from superposition.experiments.runner import run_experiment
from superposition.models import get_activation
from superposition.runs import list_runs

pd.set_option("display.max_colwidth", 120)

REUSE = True  # reuse your latest quick-profile run if there is one; False always re-runs


def quick_run(name: str):
    """Your latest complete quick-profile run of ``name`` with its figures, or a fresh one."""
    if REUSE:
        for run in reversed(list_runs(name)):
            quick = yaml.safe_load(run.config_path.read_text())["profile"] == "quick"
            if quick and any(run.figures_dir.glob("*.png")):
                print(f"reusing run {run.run_id}")
                return run
    return run_experiment(name, "quick", device="auto")


def rows(run) -> dict:
    """The run's report rows, keyed by (group, id)."""
    return {(c["group"], c["id"]): c["value"] for c in run.read_acceptance()["criteria"]}

## 1. Six output activations

E02 trains the ReLU output model x′ = ReLU(WᵀWx + b). E12 swaps the ReLU for another σ and
keeps everything else. Section 3 of notebook 03 showed why the ReLU matters: on a 1-sparse
input, output j sees Wⱼ·Wᵢ xᵢ + bⱼ, and the ReLU turns every negative value into a free 0.
That is what makes an antipodal pair cheap (p. 13).

**Predict** before running the next cell. For each activation:
- Is negative interference free, nearly free, or costly?
- Can the output be exactly 0, the target of an inactive feature?

In [ ]:
u = torch.linspace(-3, 3, 301, dtype=torch.float64)
fig, ax = plt.subplots(figsize=(6, 3.5))
for name in DEFAULT_ACTIVATIONS:
    ax.plot(u, get_activation(name)(u), label=name)
ax.axhline(0, color="0.8", lw=0.8)
ax.axvline(0, color="0.8", lw=0.8)
ax.set_xlabel("pre-activation u")
ax.set_ylabel("σ(u)")
ax.set_ylim(-0.6, 3.1)
ax.legend()
plt.show()

pd.DataFrame(
    {
        name: {
            "σ(−1)": float(get_activation(name)(torch.tensor(-1.0))),
            "σ(0)": float(get_activation(name)(torch.tensor(0.0))),
            "min over [−3, 3]": float(get_activation(name)(u).min()),
        }
        for name in DEFAULT_ACTIVATIONS
    }
).T.round(4)

- **ReLU:** negative interference costs nothing, and an inactive feature's output is exactly 0
  whenever its pre-activation is ≤ 0.
- **Leaky ReLU** (slope 0.01): the same, up to a 1% leak.
- **GELU and SiLU** dip below zero (to about −0.17 and −0.28). A negative pre-activation of the
  wrong size gives a negative output, and that is an error against a target of 0. Negative
  interference is no longer free.
- **Softplus and sigmoid** never reach 0, and σ(0) is 0.69 and 0.5. To output nearly 0 they
  need a very negative bias, and then they need large weights to respond to the feature at all.

## 2. One cell, exactly

`superposition.theory` computes a weight family's expected loss exactly, summing over sparsity
patterns as on p. 13. For the ReLU this has a closed form (notebook 03). For any other σ, E12
uses Gauss–Legendre quadrature split where σ's argument changes sign (docs/DECISIONS.md D-601).

Take one cell: both features of importance 1, density 1 − S = 0.1. **Predict** which
activation pays most for an antipodal pair, and whether any of them would rather drop the
second feature.

In [ ]:
settings = ActivationFunctionsConfig().theory  # E12's search box (D-602)
importance, density = np.array([[1.0, 1.0]]), 0.1
families = (*theory.N2M1_FAMILIES, theory.N2M1_CONFUSED)
table = {}
for name in DEFAULT_ACTIVATIONS:
    optimizer = settings.optimizer(name)
    table[name] = {
        f"{f.name} ({f.label})": float(
            theory.optimize_family(f, importance, density, optimizer).loss[0]
        )
        for f in families
    }
losses = pd.DataFrame(table).T
losses["winner"] = losses.idxmin(axis=1)
losses

Every column is the *optimal* loss of one weight family. Within one cell the losses are
comparable across activations, because they measure the same task. Footnote 17 only forbids
comparing across sparsities.

- The ReLU and the leaky ReLU are almost indistinguishable.
- The smooth activations pay more for every family, because they cannot output an exact 0. That
  extra cost is also why the winning family can change from one activation to another, near the
  boundaries.

## 3. The diagrams (E12)

E12 runs E02's whole n = 2, m = 1 grid (8 × 8 cells in the quick profile) for every activation:
ten models per cell, the highest-loss one dropped, the majority label of the other nine. The
starting weights are identical for every activation, and each activation's theoretical diagram
uses its own exact losses.

**Predict:** which activations will show *less* superposition than the ReLU, and which more?

In [ ]:
run = quick_run("activation_functions")
display(Image(filename=str(run.figures_dir / "diagrams_n2_m1.png")))
display(Image(filename=str(run.figures_dir / "shares_n2_m1.png")))

The figure has three rows:
- the vote of the trained models;
- each activation's own theory;
- the cells where the vote differs from the ReLU's.

## 4. Landscape or optimization?

A difference between two activations' *empirical* diagrams can have two causes:
- **The landscape:** the theory moves too, so a different family really is optimal.
- **Optimization:** the theory stays put, but the trained models land elsewhere.

The report rows separate the two:
- `best_model_vs_paper_families` asks whether the lowest-loss model of each cell agrees with the
  theory even where the vote does not;
- `best_model_over_theory` asks how close training gets to the optimal loss.

In [ ]:
report = rows(run)
summary = {}
for name in DEFAULT_ACTIVATIONS:
    group = f"n2_m1 · {name}"
    share = report[("n2_m1", "superposition_share")]["shares"][name]
    agreement = report[(group, "agreement_with_own_theory")]
    gap = report[(group, "losses")]["best_model_over_theory"]
    summary[name] = {
        "superposition (vote)": share["empirical"],
        "superposition (theory)": share["theory"],
        "vote agrees with theory": agreement["paper_families"],
        "best model agrees with theory": agreement["best_model_vs_paper_families"],
        "best model / theory, median": gap["median"],
        "best model / theory, max": gap["max"],
    }
pd.DataFrame(summary).T

Read across each row. Where the vote shows less superposition than the theory, but the best
model agrees with the theory, the models that voted otherwise are stuck.

E02 found one source of stuck models (`initial_sign`, review finding R1-02). In one dimension
the two weights must pass through zero to become an antipodal pair, so a model whose weights
*start* with the same sign often never gets there. E12 reports the same split for every
activation, from the same starting weights. Runs made before that row existed do not have it.

In [ ]:
signs = {name: report.get((f"n2_m1 · {name}", "initial_sign")) for name in DEFAULT_ACTIVATIONS}
pd.DataFrame(signs).T if any(signs.values()) else "this run predates the initial_sign row"

## 5. The paper's open questions (pp. 45–46)

The paper ends with a list of questions. docs/STUDY_GUIDE.md turns them into a menu of
extension projects, each with a first experiment you can run with this library. Four that build
directly on what you have done here:

- **Can we change the phase diagram so that features avoid superposition?** The paper names
  L1 on activations, adversarial training and the activation function. E11 and E12 are first
  steps; try them on one phase-diagram cell (exercise 4).
- **Is there a closed-form solution beyond n = 2, m = 1?** `theory.py` solves restricted
  families exactly. Try a two-scale antipodal family, which review finding R1-03 suggests beats
  the one-scale family in many cells.
- **How does superposition change with correlated or anticorrelated features?** Start from E04
  (`data.CorrelatedSets`, `data.AnticorrelatedSets`).
- **How much computation can happen in superposition?** Start from E08 and E09's absolute-value
  models and change the target function.

## Exercises

**1.** Register a leaky ReLU with slope 0.3 and compute the antipodal family's optimal loss at
the cell of section 2. How does it compare with slope 0.01 and with the ReLU?

<details><summary>Solution</summary>

```python
import torch.nn.functional as F
from superposition.models import ACTIVATIONS, register_activation

if "leaky_relu_0.3" not in ACTIVATIONS:
    register_activation("leaky_relu_0.3", lambda u: F.leaky_relu(u, 0.3))
pair = theory.N2M1_FAMILIES[2]  # W ⊥ [1, 1]: an antipodal pair
for name in ("relu", "leaky_relu", "leaky_relu_0.3"):
    loss = theory.optimize_family(pair, importance, density, settings.optimizer(name)).loss[0]
    print(f"{name:15s} {loss:.5f}")
```

The leak lets negative interference through as a negative output, so the pair pays for it.
At slope 0.3 the price is no longer negligible. Negative interference is free only when the
activation clips it to exactly 0.
</details>

**2.** Which cells change label between the ReLU's and GELU's empirical diagrams? Are they next
to a theoretical boundary?

<details><summary>Solution</summary>

```python
cells = run.read_table("cells")
relu = cells[cells["activation"] == "relu"].reset_index(drop=True)
gelu = cells[cells["activation"] == "gelu"].reset_index(drop=True)
changed = relu["empirical"] != gelu["empirical"]
pd.DataFrame(
    {
        "density": relu["density"], "importance": relu["importance"],
        "relu": relu["empirical"], "gelu": gelu["empirical"],
        "gelu theory": gelu["theory"], "near a gelu boundary": gelu["near_theory_boundary"],
    }
)[changed]
```

Compare the `gelu` column with `gelu theory`. Where they agree, GELU's landscape really moved
the boundary. Where they disagree, the vote is off GELU's own optimum, which is optimization.
</details>

**3.** Train the ten models of one cell with a sigmoid output for three times as many steps.
Does the best model's exact loss approach the theory?

<details><summary>Solution</summary>

```python
from superposition.config import TrainConfig
from superposition.data import IndependentSparse
from superposition.models import TiedModel
from superposition.train import train_from_config

p, extra = 0.1, 1.0
imp = torch.tensor([1.0, extra])
model = TiedModel(2, 1, 10, out_act="sigmoid", seed=0)
train_from_config(
    model, IndependentSparse(p, 10, 2), imp, TrainConfig(steps=30_000, batch_size=256), seed=0
)
exact = theory.tied_output_expected_losses(
    model.W.detach().double().numpy(), model.b.detach().double().numpy(), imp.numpy(), p, "sigmoid"
)
optimum = min(
    theory.optimize_family(f, imp.numpy(), p, settings.optimizer("sigmoid")).loss[0]
    for f in families
)
print(f"best trained {exact.min():.5f} vs theory {optimum:.5f}")
```

The sigmoid needs large weights and very negative biases. At lr 1e-3, Adam moves each weight by
about 1e-3 per step, so 10,000 steps can be too few in some cells. When the gap shrinks with
more steps, the difference was optimization, not the landscape.
</details>

**4.** (Open question, p. 45.) Add an L1 penalty on h to one phase-diagram cell where the ReLU
model stores the extra feature in superposition. At what λ does the extra feature stop being
superposed, and what does it become?

<details><summary>Solution</summary>

```python
from superposition.experiments.phase_change import extra_feature_regime

p, extra = 0.03, 1.0
for lam in (0.0, 0.01, 0.03, 0.1, 0.3):
    model = TiedModel.from_kind("relu_output", 2, 1, 10, seed=0)
    config = TrainConfig(steps=10_000, batch_size=256, l1_hidden=lam)
    train_from_config(model, IndependentSparse(p, 10, 2), torch.tensor([1.0, extra]), config, seed=0)
    labels, _, _ = extra_feature_regime(model.W.detach(), feature=1)
    print(lam, pd.Series(labels).value_counts().to_dict())
```

An L1 penalty on h makes every unit of hidden activity cost something. In m = 1 both features
share the one hidden unit, so the cheaper solution is eventually to drop the less useful one.
That is p. 42's "killing features" rather than a change of geometry (compare E11).
</details>